# DS2002 · Pandas Challenge

**Lab — 2026-09-18 · Fall 2026**  

---

## Lab 04 — Pandas Challenge

Four hundred generated orders. Each question builds toward a demand report you could hand a vendor.

The data is seeded, so everyone's numbers should match. That is deliberate: if your total revenue differs from your neighbor's, one of you has a bug, and the assertions at the end will tell you which.

Every answer needs the number **and** a sentence saying what it means. A cell that prints `4218.5` with no interpretation is half an answer.

In [17]:
import pandas as pd, numpy as np
rng = np.random.default_rng(4)
n = 400
df = pd.DataFrame({
    'vendor_id': rng.choice(['V-01','V-05','V-10','V-18'], n),
    'category': rng.choice(['Food','Merch','RainGear','Drink'], n, p=[.5,.2,.1,.2]),
    'qty': rng.integers(1, 4, n),
    'price': rng.choice([4.5, 6.0, 7.5, 12.0, 24.0], n),
})
df.head()

,vendor_id,category,qty,price
0,V-10,Drink,2,24.0
1,V-18,RainGear,1,12.0
2,V-18,Drink,3,4.5
3,V-10,Food,2,12.0
4,V-18,Drink,3,7.5


### Q1 — Add `revenue`, then report total revenue and total units.

*Expected: 400 rows, and revenue should land between $8,000 and $9,000.*

In [18]:
# TODO
# revenue is the number of items * the price of each item per vendor. so, we add revenue to the dataframe as df['qty'] * df['price']
df['revenue'] = df['qty'] * df['price']
# total revenue is the sum of each column value for each row
print(f"The number of rows is {len(df)}")
print(f"The total revenue for all vendors is ${df['revenue'].sum()}")
# since every row has a different qty, we want to sum up the qty values not just return the number of rows
print(f"The total number of units sold is {df['qty'].sum()}")


The number of rows is 400
The total revenue for all vendors is $8520.0
The total number of units sold is 783


### Q2 — Revenue by category, highest to lowest.

Include the share of total as a percentage in the same table.

In [19]:
# TODO
by_category = pd.DataFrame(df.groupby('category')['revenue'].sum())
by_category = by_category.sort_values('revenue', ascending=False)
by_category['percentage_share'] = (by_category['revenue']/by_category['revenue'].sum()) * 100
print(f"Food generated the highest revenue at ${by_category.loc['Food', 'revenue']}, account for around {by_category.loc['Food', 'percentage_share']:.1f}% of the total share")
by_category

Food generated the highest revenue at $4293.0, account for around 50.4% of the total share


,revenue,percentage_share
category,,
Food,4293.0,50.387324
Merch,1771.5,20.792254
Drink,1554.0,18.239437
RainGear,901.5,10.580986


### Q3 — Which vendor has the highest *average* order revenue?

Report the average alongside the order count for each vendor. A high average on twelve orders is a different claim from a high average on two hundred.

In [20]:
# TODO
vendor_orders = df.groupby('vendor_id').agg(
    orders=('qty', 'count'),
    revenue=('revenue', 'sum')
)

vendor_orders['avg_per_order'] = vendor_orders['revenue'] / vendor_orders['orders']
print(f"Vendor V-01 has the highest average order revenue at approximately ${vendor_orders.loc['V-01', 'avg_per_order']:.1f} across {vendor_orders.loc['V-01', 'orders']:.1f} orders for total revenue of ${vendor_orders.loc['V-01', 'revenue']}")
vendor_orders

Vendor V-01 has the highest average order revenue at approximately $22.6 across 94.0 orders for total revenue of $2124.0


,orders,revenue,avg_per_order
vendor_id,,,
V-01,94,2124.0,22.595745
V-05,93,1914.0,20.580645
V-10,105,2133.0,20.314286
V-18,108,2349.0,21.750000


### Q4 — What share of revenue comes from Merch?

Print it as a percentage rounded to one decimal.

In [21]:
# TODO

print(f"The share of revenue that comes from merch is {by_category.loc['Merch', 'percentage_share']:.1f}%")

The share of revenue that comes from merch is 20.8%


### Q5 — Join in the vendor names.

The frame only has `vendor_id`. Merge the lookup below so your report is readable.

**Requirements:** left join, `validate='many_to_one'`, and prove the row count and revenue total did not change. One vendor id in the orders is not in this lookup — find it, and decide what to do about it.

In [22]:
vendor_names = pd.DataFrame({
    'vendor_id': ['V-01', 'V-05', 'V-10'],
    'vendor_name': ['Hoos Burgers', 'Rotunda Tacos', 'Cav Merch North'],
})

# TODO: merge, validate, and report the unmatched vendor
joined = df.merge(vendor_names, on='vendor_id', how='left', validate='many_to_one', indicator=True)
print(f"Row count before: {len(df)}, row count after: {len(joined)}")
print(f"Revenue before: ${df['revenue'].sum()}, revenue after: ${joined['revenue'].sum()}")
unmatched = joined[joined['_merge'] == 'left_only']
unmatched_vendor = unmatched['vendor_id'].unique()[0]
print(f"The unmatched vendor is {unmatched_vendor}.")
joined['vendor_name'] = joined['vendor_name'].fillna('Unknown vendor')
joined = joined.drop(columns='_merge')

joined

Row count before: 400, row count after: 400
Revenue before: $8520.0, revenue after: $8520.0
The unmatched vendor is V-18.


,vendor_id,category,qty,price,revenue,vendor_name
0,V-10,Drink,2,24.0,48.0,Cav Merch North
1,V-18,RainGear,1,12.0,12.0,Unknown vendor
2,V-18,Drink,3,4.5,13.5,Unknown vendor
3,V-10,Food,2,12.0,24.0,Cav Merch North
4,V-18,Drink,3,7.5,22.5,Unknown vendor
...,...,...,...,...,...,...
395,V-18,Merch,1,12.0,12.0,Unknown vendor
396,V-01,Merch,2,24.0,48.0,Hoos Burgers
397,V-10,Food,3,7.5,22.5,Cav Merch North
398,V-18,Merch,2,24.0,48.0,Unknown vendor


**The unmatched vendor, and what I did about it:** The unmatched vendor was V-18. Similar to the studio checkpoint, I kept the unmatched vendor but named it to Unknown rather than dropping it completely in order to keep the total revenue accurate and flag the unmatched vendor.

### Q6 — A pivot table: vendors down the side, categories across the top, revenue in the cells.

Add row and column totals so it reads as a report rather than a grid of numbers.

In [23]:
# TODO
pivot = pd.pivot_table(joined, index = 'vendor_name', columns = 'category', values = 'revenue', aggfunc = 'sum', margins = True, margins_name='Total')
print("The table shows that the unknown vendor generated the highest revenue out of all the vendors, and Food had the highest revenue out of all categories.")
pivot

The table shows that the unknown vendor generated the highest revenue out of all the vendors, and Food had the highest revenue out of all categories.


category,Drink,Food,Merch,RainGear,Total
vendor_name,,,,,
Cav Merch North,502.5,1054.5,400.5,175.5,2133.0
Hoos Burgers,171.0,1338.0,373.5,241.5,2124.0
Rotunda Tacos,298.5,882.0,489.0,244.5,1914.0
Unknown vendor,582.0,1018.5,508.5,240.0,2349.0
Total,1554.0,4293.0,1771.5,901.5,8520.0


### Q7 — Validate your work

**TODO:** uncomment and make these pass. Assign your results to the named variables as you go.

In [24]:
assert len(df) == 400
assert 8000 < df['revenue'].sum() < 9000, df['revenue'].sum()
assert abs(by_category['revenue'].sum() - df['revenue'].sum()) < 0.01
assert len(joined) == len(df), 'the vendor merge changed the row count'
print('checks passed.')

checks passed.


### Write-up

**a)** What would you tell these vendors to do differently next game? One paragraph, with at least two numbers from your report in it.

**b)** Which of your seven answers is the least trustworthy, and why? Point at a specific weakness — a small group size, an unmatched vendor, a category that is really two things.

a) One thing I would tell my vendors to do next game is to probably invest less in rain gear. Throughout the steps of creating the report, we see that rain gear consistently is not as important to customers of these vendors. For example, in the revenue by category table, we see that rain gear composes only 10% of the total revenue. Instead, I would focus that budget they have into the food category instead. We see that in the same table, food makes up 50% of the entire revenue, making up over $4000. If vendors instead invested in food more, they would definitely see higher returns.

b) I think that my pivot table for Q6 is the least trustworthy mostly because of the unmatched vendor V-18, which accounts for the highest per-vendor revenue out of all four. However, the problem is that it makes any vendor comparison unreliable since V-18 could easily be a system glitch or a duplicate/typo. As a result, the actual performance of the vendors could be inaccurate since we don't know what V-18 is.

